In [1]:
import pandas as pd
from datasets import Dataset
df = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/evaluation_data.csv')


/home/wagyu0923/miniconda3/envs/exaone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os
sys.path.append('/home/wagyu0923/project/Document_Analyzer')
from pipeline.document_loader import DocumentLoader
from pipeline.chunker import Chunker
from pipeline.embedder import Embedder
from pipeline.vector_retriever import VectorRetriever
from pipeline.generator import Generator
import config
from tkinter import filedialog

def setup_pipeline():
    chunker = Chunker(
        chunk_size = config.CHUNK_SIZE,
        overlap_size = config.OVERLAP_SIZE
    )
    print('Chunking Complete')
    embedder = Embedder(
        model_name = config.EMBEDDING_MODEL
    )
    print('Embedding Complete')
    retriever =VectorRetriever(
        db_path = config.DB_PATH,
        model_name = config.EMBEDDING_MODEL,
        collection_name = config.COLLECTION_NAME
    )
    print('Retrieving Coplete')
    generator = Generator(
        model_name = config.LLM_NAME,
        options = config.DEFAULT_OLLAMA_OPTIONS
    )
    return chunker, embedder, retriever, generator

def run_indexing(file_path, chunker, embedder, retriever):
    loader = DocumentLoader(file_path = file_path)
    document = loader.load()
    chunks = chunker.chunking(document)
    embedded_chunks = embedder.embed_documents(chunks)
    file_name = os.path.basename(file_path)
    retriever.add_documents(embedded_chunks, file_name)

chunker, embedder, retriever, generator = setup_pipeline()

file_path = '/home/wagyu0923/project/Document_Analyzer/pdf_files/[세토피아][정정]반기보고서(2025.09.09).pdf'
run_indexing(file_path, chunker, embedder, retriever)

Chunking Complete
Embedding Complete
Retrieving Coplete


In [3]:
import json
dataset = df.copy()
for index, query in enumerate(df['user_input']):
    retrieved_data = retriever.retrieve(query)
    outputs = generator.generate(retrieved_data, query)
    try:
        outputs = json.loads(outputs)
    except json.JSONDecodeError:
        print(f'JSON Decode Error at index {index}. Skipping.') 
        continue 
    if 'used_context' not in outputs.keys():
        outputs['used_context'] = []
    elif 'answer' not in outputs.keys():
        outputs['answer'] = ''
    dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
    dataset.loc[index, 'response'] = outputs['answer']
    print(f'progress : {index+1}/{len(df)}')




/tmp/ipykernel_28567/3895652976.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '대표이사 (성 명) 서상철' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
/tmp/ipykernel_28567/3895652976.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The current CEO of Setopia Co., Ltd. is 서상철.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'response'] = outputs['answer']


progress : 1/30
progress : 2/30
progress : 3/30
progress : 4/30
progress : 5/30
progress : 6/30
progress : 7/30
progress : 8/30
progress : 9/30
progress : 10/30
progress : 11/30
progress : 12/30
progress : 13/30
progress : 14/30
progress : 15/30
progress : 16/30
progress : 17/30
progress : 18/30
progress : 19/30
progress : 20/30
progress : 21/30
progress : 22/30
progress : 23/30
progress : 24/30
progress : 25/30
progress : 26/30
progress : 27/30
progress : 28/30
progress : 29/30
progress : 30/30


In [4]:
dataset

,user_input,retrieved_contexts,response,reference
0,"Who is the current CEO of Setopia Co., Ltd.?",대표이사 (성 명) 서상철,"The current CEO of Setopia Co., Ltd. is 서상철.","The current CEO of Setopia Co., Ltd. is Sang-c..."
1,"What was the company name of Setopia Co., Ltd....",2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호명...,It was (주)마이더스AI.,"It was Midas AI Co., Ltd."
2,"When was the merger date for Setopia Co., Ltd....",,I cannot answer the question based on the prov...,"The merger date was January 2, 2023."
3,What is the main product of Setopia's steel bu...,"철강사업 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 있으며...",Setopia’s steel business mainly sells stainles...,"The main product is STS 201, and it holds the ..."
4,The distribution business Setopia entered into...,2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을 체결,ELFBAR (엘프바),It was an exclusive domestic distributorship c...
5,Who was the auditor for the 11th fiscal period...,삼일회계법인 의견거절,"The auditor was 삼일회계법인, and the audit opinion ...","The auditor was Samil PwC, and the audit opini..."
6,What happened to the contract for the acquisit...,해지 하는 것으로 결정하게 되었습니다 계약상대방 (주)에스에이코퍼레이션 2. 자산구...,The contract for acquiring the land and buildi...,"The contract was terminated on January 20, 2025."
7,What was the ratio of the capital reduction wi...,"13,332,272 88.11% 합 계 15,131,870 100.00%",The ratio was 88.11%.,A 5-to-1 capital reduction was completed.
8,"Who is the largest shareholder of Setopia Co.,...","(주)에스에이코퍼레이션 1,042,986 6.89%","The largest shareholder is (주)에스에이코퍼레이션, holdi...","The largest shareholder is SA Corporation Co.,..."
9,"How much was the fine imposed on Setopia Co., ...",,I cannot answer the question based on the prov...,A fine of 270 million KRW was imposed related ...


In [5]:
dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(
    lambda x: []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == ""
    else (json.loads(x) if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
          else (x if isinstance(x, list) else [x]))
)

if "reference" in dataset.columns:
    dataset["reference"] = dataset["reference"].apply(
        lambda x: "" if x is None or (isinstance(x, float) and pd.isna(x))
        else (x if isinstance(x, str) else "\n\n".join(map(str, x)))
    )

In [12]:
dataset.to_csv('response_data.csv')

In [ ]:
# ===== RAGAS 평가: Ollama + SentenceTransformer (LangChain 없음) =====
import asyncio
from dataclasses import dataclass
from typing import Any, List, Optional

import ollama
from datasets import Dataset
from sentence_transformers import SentenceTransformer

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import BaseRagasLLM
from ragas.embeddings import BaseRagasEmbeddings
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)

# -------------------------------------------------------------------
# 1) RAGAS가 기대하는 LLM 결과 포맷 (간단한 구조체)
# -------------------------------------------------------------------
@dataclass
class SimpleGeneration:
    text: str

@dataclass
class SimpleLLMResult:
    # ragas 내부에서 result.generations[0][0].text 이런 식으로 접근
    generations: List[List[SimpleGeneration]]


# -------------------------------------------------------------------
# 2) Ollama용 RAGAS LLM 래퍼
# -------------------------------------------------------------------
class OllamaRagasLLM(BaseRagasLLM):
    def __init__(self, model: str = "llama3.1:8b-instruct-q4_K_M", host: str = "http://127.0.0.1:11434"):
        # BaseRagasLLM은 dataclass지만, run_config는 evaluate() 쪽에서 set_run_config로 주입되므로
        # 여기서는 super().__init__() 안 불러도 동작에 문제 없음
        self.model = model
        self.client = ollama.Client(host=host)

    def _to_text(self, prompt: Any) -> str:
        # ragas가 넘겨주는 prompt 타입이 다양할 수 있어서 안전하게 문자열만 뽑아주는 헬퍼
        if hasattr(prompt, "to_string"):
            return prompt.to_string()
        if hasattr(prompt, "text"):
            return prompt.text
        return str(prompt)

    def generate_text(
        self,
        prompt: Any,
        n: int = 1,
        temperature: float = 1e-8,
        stop: Optional[List[str]] = None,
        callbacks: Optional[Any] = None,
    ) -> SimpleLLMResult:
        prompt_text = self._to_text(prompt)
        messages = [{"role": "user", "content": prompt_text}]

        gens: List[List[SimpleGeneration]] = []

        # ragas는 보통 n=1로 호출하지만, 인터페이스 맞춰서 루프 유지
        for _ in range(max(n, 1)):
            resp = self.client.chat(
                model=self.model,
                messages=messages,
                options={"temperature": max(temperature, 0.0)},
            )
            text = resp["message"]["content"]

            # stop 토큰 처리
            if stop:
                for s in stop:
                    idx = text.find(s)
                    if idx != -1:
                        text = text[:idx]
                        break

            gens.append([SimpleGeneration(text=text)])

        return SimpleLLMResult(generations=gens)

    async def agenerate_text(
        self,
        prompt: Any,
        n: int = 1,
        temperature: float = 1e-8,
        stop: Optional[List[str]] = None,
        callbacks: Optional[Any] = None,
    ) -> SimpleLLMResult:
        # 비동기는 그냥 동기 generate_text를 쓰레드 풀에서 돌립니다.
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(
            None,
            self.generate_text,
            prompt,
            n,
            temperature,
            stop,
            callbacks,
        )

    def is_finished(self, response: SimpleLLMResult) -> bool:
        """
        일부 ragas 버전에서 요구하는 추상 메서드.
        여기서는 텍스트가 비어있지 않으면 완료된 것으로 간주.
        """
        try:
            for gen_list in response.generations:
                for gen in gen_list:
                    if not getattr(gen, "text", "").strip():
                        return False
            return True
        except Exception:
            return False


# -------------------------------------------------------------------
# 3) SentenceTransformer 기반 로컬 임베딩 (BaseRagasEmbeddings 구현)
# -------------------------------------------------------------------
class LocalHFEmbeddings(BaseRagasEmbeddings):
    def __init__(self, model_name: str = "intfloat/multilingual-e5-large-instruct"):
        self.model = SentenceTransformer(model_name)

    # 동기 버전
    def embed_query(self, text: str) -> List[float]:
        emb = self.model.encode([text], normalize_embeddings=True)[0]
        return emb.tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        if len(texts) == 0:
            return []
        embs = self.model.encode(texts, normalize_embeddings=True)
        return [e.tolist() for e in embs]

    # 비동기 버전 (간단히 동기 함수 재사용)
    async def aembed_query(self, text: str) -> List[float]:
        return self.embed_query(text)

    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.embed_documents(texts)


# -------------------------------------------------------------------
# 4) 인스턴스 생성
# -------------------------------------------------------------------
llm = OllamaRagasLLM(
    model="gpt-oss:20b",          # ollama 모델 이름
    host="http://127.0.0.1:11434" # 기본값이면 바꿀 필요 없음
)

embeddings = LocalHFEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct"
)

# -------------------------------------------------------------------
# 5) pandas DataFrame -> HuggingFace Dataset 변환
#    (dataset 변수는 이미 위에서 만들어졌다고 가정)
# -------------------------------------------------------------------
hf_ds = Dataset.from_pandas(
    dataset[["user_input", "retrieved_contexts", "response", "reference"]]
)

# -------------------------------------------------------------------
# 6) 사용할 메트릭 정의
# -------------------------------------------------------------------
metrics = [context_precision, context_recall, faithfulness, answer_relevancy]

# -------------------------------------------------------------------
# 7) 평가 실행
# -------------------------------------------------------------------
run_config = RunConfig(
    timeout=180,   # 한 샘플당 최대 180초까지 기다림 (원래 기본값과 비슷)
    max_workers=2, # 동시에 2개 샘플만 평가해서 GPU 안터지게
    max_retries=1, # 재시도 너무 많이 안 하도록
)
try:
    result = evaluate(
        dataset=hf_ds,
        metrics=metrics,
        llm=llm,
        embeddings=embeddings,
        run_config=run_config
    )
except Exception:
    # 설치된 ragas 버전에 따라 column_map을 요구할 수 있어서 예비 처리
    column_map = {
        "question": "user_input",
        "contexts": "retrieved_contexts",
        "answer": "response",
        "ground_truth": "reference",
    }
    result = evaluate(
        dataset=hf_ds,
        metrics=metrics,
        llm=llm,
        embeddings=embeddings,
        column_map=column_map,
        run_config=run_config
    )

print(result)          # 전체 평균 점수
result_df = result.to_pandas()
result_df.head()       # 샘플별 점수 확인용
# 필요하면 저장
# result_df.to_csv("ragas_eval_result.csv", index=False)

Evaluating:   3%|▎         | 4/120 [03:45<1:30:59, 47.06s/it] LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[3]: ValidationError(1 validation error for EmbeddingUsageEvent
model
  Input should be a valid string [type=string_type, input_value=SentenceTransformer(
  (0...e})
  (2): Normalize()
), input_type=SentenceTransformer]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type)
Evaluating:   8%|▊         | 9/120 [04:55<33:29, 18.11s/it]  LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[7]: ValidationError(1 validation error for EmbeddingUsageEvent
model
  Input should be a valid string [type=string_type, input_value=SentenceTransformer(
  (0...e})
  (2): Normalize()
), input_type=SentenceTransformer]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type)
Evaluating:  10%|█         | 12/120 [06:05<40:15, 22.36s/it]LLM

In [10]:
result_df

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,"Who is the current CEO of Setopia Co., Ltd.?",[대표이사 (성 명) 서상철],"The current CEO of Setopia Co., Ltd. is 서상철.","The current CEO of Setopia Co., Ltd. is Sang-c...",1.0,1.0,NaN,NaN
1,"What was the company name of Setopia Co., Ltd....",[2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호...,It was (주)마이더스AI.,"It was Midas AI Co., Ltd.",1.0,1.0,1.0,NaN
2,"When was the merger date for Setopia Co., Ltd....",[],I cannot answer the question based on the prov...,"The merger date was January 2, 2023.",0.0,0.0,NaN,NaN
3,What is the main product of Setopia's steel bu...,"[철강사업 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 있으...",Setopia’s steel business mainly sells stainles...,"The main product is STS 201, and it holds the ...",0.0,0.0,NaN,NaN
4,The distribution business Setopia entered into...,[2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을...,ELFBAR (엘프바),It was an exclusive domestic distributorship c...,1.0,1.0,NaN,NaN
5,Who was the auditor for the 11th fiscal period...,[삼일회계법인 의견거절],"The auditor was 삼일회계법인, and the audit opinion ...","The auditor was Samil PwC, and the audit opini...",1.0,1.0,NaN,NaN
6,What happened to the contract for the acquisit...,[해지 하는 것으로 결정하게 되었습니다 계약상대방 (주)에스에이코퍼레이션 2. 자산...,The contract for acquiring the land and buildi...,"The contract was terminated on January 20, 2025.",1.0,0.0,NaN,NaN
7,What was the ratio of the capital reduction wi...,"[13,332,272 88.11% 합 계 15,131,870 100.00%]",The ratio was 88.11%.,A 5-to-1 capital reduction was completed.,0.0,0.0,NaN,NaN
8,"Who is the largest shareholder of Setopia Co.,...","[(주)에스에이코퍼레이션 1,042,986 6.89%]","The largest shareholder is (주)에스에이코퍼레이션, holdi...","The largest shareholder is SA Corporation Co.,...",1.0,1.0,NaN,NaN
9,"How much was the fine imposed on Setopia Co., ...",[],I cannot answer the question based on the prov...,A fine of 270 million KRW was imposed related ...,0.0,0.0,NaN,NaN
